# Basic Decision Tree from Scratch - Classification (Gini) - Optimization

In this notebook, we are going to optimize the Gini search algorithm we had.

## Dataset

Dataset: This is a makeup dataset that describe how much study hours a student put in and whether if they will either pass or fail their exam

| Study Hours  | Pass Exam |
| ----- | ----- |
| 1    | No    |
| 2    | No    |
| 3    | No    |
| 4    | No    |
| 5    | Yes   |
| 6    | Yes   |
| 7    | Yes   |
| 8    | Yes   |
| 9    | Yes   |
| 10    | Yes   |
| 11    | Yes   |

In [1]:
import pandas as pd
import numpy as np
import json

In [2]:
df = pd.DataFrame({'studied': [1,2,3,4,5,6,7,8,9,10,11,12], 
        'passed': [0,0,0,0,1,1,1,1,1,1,1,1]})

In [3]:
df

,studied,passed
0,1,0
1,2,0
2,3,0
3,4,0
4,5,1
5,6,1
6,7,1
7,8,1
8,9,1
9,10,1


## Gini Formula

### Gini Impurity Formula

For a node:

$$\text{Gini} = 1 - \sum_{k=1}^{K} p_k^2$$


Where:

- K = number of classes
- $p_k$ = proportion of class (k) in the node


#### Binary classification Example (0/1 case)

$$\text{Gini} = 1 - (p_0^2 + p_1^2)$$

Where:

- $p_0 = \frac{0}{N}$
- $p_1 = \frac{1}{N}$

**Intuition**

| Case           | Gini                          |
| -------------- | ----------------------------- |
| all same class | 0 (pure)                      |
| 50/50 split    | 0.5 (max impurity for binary) |

---

### Weighted Gini (for a split)

When you split a node into LEFT and RIGHT:

$$ \text{Weighted Gini} = \frac{N_L}{N} \cdot Gini(L) + \frac{N_R}{N} \cdot Gini(R)$$

Where:

- $N_L$ = number of samples in left node
- $N_R$ = number of samples in right node
- $N = N_L + N_R$


**Intuition**

Weighted Gini means:

> "how impure are the children, adjusted by their size?"

So:

- big bad child → matters a lot
- small bad child → matters less

### Gini split procedure:
- Sort feature values
- For each adjacent pair:
    - compute midpoint threshold
- For each threshold:
    - split data
    - compute weighted Gini
- Choose best threshold

## Midpoint Optimization

In [4]:
# compute mid points between values of studied
mid_points = []
for i in range(len(df['studied']) - 1):
    mid_point = (df['studied'][i] + df['studied'][i+1]) / 2
    # [i] refer to the fist value and [i+1] refer to the second value and so on
    mid_points.append(float(mid_point))

print(mid_points)

[1.5, 2.5, 3.5, 4.5, 5.5, 6.5, 7.5, 8.5, 9.5, 10.5, 11.5]


In [5]:
x = df['studied'].values
x

array([ 1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12])

In [6]:
print(x[:-1])
print(x[1:])

[ 1  2  3  4  5  6  7  8  9 10 11]
[ 2  3  4  5  6  7  8  9 10 11 12]


In [7]:
mid_points = (x[:-1] + x[1:]) / 2
mid_points

array([ 1.5,  2.5,  3.5,  4.5,  5.5,  6.5,  7.5,  8.5,  9.5, 10.5, 11.5])

## Current Gini Algorithm

In [8]:
def gini(y):
    '''
    Compute the Gini impurity for a set of labels.
    y: array-like of shape (n_samples,) - target labels for the samples in the node
    return: float - Gini impurity of the node
    '''
    if len(y) == 0:
        return 0
    p0 = (y == 0).sum() / len(y)
    p1 = (y == 1).sum() / len(y)

    #print(f"p0: {p0:.2f}, p1: {p1:.2f}")

    gini = 1 - (p0**2 + p1**2)
    return gini

In [9]:
# weighted gini impurity
def weighted_gini(df_left, df_right):
    total_samples = len(df_left) + len(df_right)
    gini_left = gini(df_left['passed'])
    gini_right = gini(df_right['passed'])
    
    weighted_gini = (len(df_left) / total_samples) * gini_left + (len(df_right) / total_samples) * gini_right
    return weighted_gini

In [10]:
def best_gini_split(df):
    best_gini = float('inf')
    best_split_value = None
    df_sorted = df.sort_values(by='studied').reset_index(drop=True)
    
    for i in range(len(df_sorted['studied']) - 1):
        mid_point = (df_sorted['studied'][i] + df_sorted['studied'][i+1]) / 2
        df_left = df_sorted[df_sorted['studied'] <= mid_point]
        df_right = df_sorted[df_sorted['studied'] > mid_point]
        
        current_gini = weighted_gini(df_left, df_right)
        
        print(f"Midpoint: {mid_point}, Weighted Gini: {current_gini:.2f}")  
        
        if current_gini < best_gini:
            best_gini = current_gini
            best_split_value = mid_point
            
    return best_split_value, best_gini

In [11]:
best_split_value, best_gini = best_gini_split(df)
print(f"Best split value: {best_split_value}, Best Gini impurity: {best_gini:.2f}")

Midpoint: 1.5, Weighted Gini: 0.36
Midpoint: 2.5, Weighted Gini: 0.27
Midpoint: 3.5, Weighted Gini: 0.15
Midpoint: 4.5, Weighted Gini: 0.00
Midpoint: 5.5, Weighted Gini: 0.13
Midpoint: 6.5, Weighted Gini: 0.22
Midpoint: 7.5, Weighted Gini: 0.29
Midpoint: 8.5, Weighted Gini: 0.33
Midpoint: 9.5, Weighted Gini: 0.37
Midpoint: 10.5, Weighted Gini: 0.40
Midpoint: 11.5, Weighted Gini: 0.42
Best split value: 4.5, Best Gini impurity: 0.00


## Sliding Windows Optimization

Pseudo code:

- Sort feature
- Initialize LEFT counts to zero
- Initialize RIGHT counts to total class counts
- Move one sample at a time from RIGHT to LEFT
- Update counts
- Compute weighted Gini
- Track best boundary
- Convert best boundary to midpoint threshold

In [12]:
def gini(negative, positive):
    '''
    Compute the Gini impurity for a set of labels.
    negative: refers to number of negative labels
    positive: refers to number of positive labels
    '''
    total = negative + positive
    p0 = negative / total
    p1 = positive / total

    gini = 1 - (p0**2 + p1**2)
    return gini

In [13]:
def gini_search_window(df):
    # initialize best_gini values
    best_gini = float('inf')
    best_split_value = None

    # Step 1: Sort the dataframe by the 'studied' column
    sorted_df = df.sort_values(by='studied')
    x = sorted_df['studied'].values
    y = sorted_df['passed'].values
    n = len(y)

    # Step 2: Initialize LEFT Node counts
    left_total = 0
    left_positive = 0
    left_negative = 0

    right_total = n
    right_positive = (y == 1).sum()
    right_negative = (y == 0).sum()

    # Step 3: Move one sample from RIGHT to LEFT
    # Step 4: Update counts
    for i in range(n-1):
        left_total += 1
        right_total -= 1
        if y[i] == 1:
            left_positive += 1
            right_positive -= 1
        else:
            left_negative += 1
            right_negative -= 1
    

        # Compute weighted gini
        left_gini = gini(left_positive, left_negative)
        right_gini = gini(right_positive, right_negative)
        weighted_gini = ((left_total/n) * left_gini) + ((right_total/n) * right_gini)
        #print("index:",i)
        print(f"Weighted Gini: {weighted_gini:.2f}")

        if weighted_gini < best_gini:
            best_gini = weighted_gini
            best_split_value = (x[i] + x[i+1]) / 2
            print("Midpoint Threshold", best_split_value)
    
    return best_split_value, best_gini



In [14]:
best_split_value, best_gini = gini_search_window(df)
print(f"Best split value: {best_split_value}, Best Gini impurity: {best_gini:.2f}")

Weighted Gini: 0.36
Midpoint Threshold 1.5
Weighted Gini: 0.27
Midpoint Threshold 2.5
Weighted Gini: 0.15
Midpoint Threshold 3.5
Weighted Gini: 0.00
Midpoint Threshold 4.5
Weighted Gini: 0.13
Weighted Gini: 0.22
Weighted Gini: 0.29
Weighted Gini: 0.33
Weighted Gini: 0.37
Weighted Gini: 0.40
Weighted Gini: 0.42
Best split value: 4.5, Best Gini impurity: 0.00


## Gini Optimization Algorithm (CUMSUM)

In [15]:
y = df['passed'].values

In [16]:
y

array([0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1])

In [17]:
def gini_show_pos_neg(df):

    # Step 1: Sort the dataframe by the 'studied' column
    sorted_df = df.sort_values(by='studied')
    x = sorted_df['studied'].values
    y = sorted_df['passed'].values
    n = len(y)

    # Step 2: Initialize LEFT Node counts
    left_total = 0
    left_positive = 0
    left_negative = 0

    right_total = n
    right_positive = (y == 1).sum()
    right_negative = (y == 0).sum()

    # Step 3: Move one sample from RIGHT to LEFT
    # Step 4: Update counts
    for i in range(n-1):
        left_total += 1
        right_total -= 1
        if y[i] == 1:
            left_positive += 1
            right_positive -= 1
        else:
            left_negative += 1
            right_negative -= 1
        
        print(f"Position {i}")
        print(f"Left Negative: {left_negative}")
        print(f"Left Positive: {left_positive}")
        print(f"Left Total: {left_total}")
    
        print(f"Right Negative: {right_negative}")
        print(f"Right Positive: {right_positive}")
        print(f"Right Total: {right_total}")
        print('---')


In [18]:
gini_show_pos_neg(df)

Position 0
Left Negative: 1
Left Positive: 0
Left Total: 1
Right Negative: 3
Right Positive: 8
Right Total: 11
---
Position 1
Left Negative: 2
Left Positive: 0
Left Total: 2
Right Negative: 2
Right Positive: 8
Right Total: 10
---
Position 2
Left Negative: 3
Left Positive: 0
Left Total: 3
Right Negative: 1
Right Positive: 8
Right Total: 9
---
Position 3
Left Negative: 4
Left Positive: 0
Left Total: 4
Right Negative: 0
Right Positive: 8
Right Total: 8
---
Position 4
Left Negative: 4
Left Positive: 1
Left Total: 5
Right Negative: 0
Right Positive: 7
Right Total: 7
---
Position 5
Left Negative: 4
Left Positive: 2
Left Total: 6
Right Negative: 0
Right Positive: 6
Right Total: 6
---
Position 6
Left Negative: 4
Left Positive: 3
Left Total: 7
Right Negative: 0
Right Positive: 5
Right Total: 5
---
Position 7
Left Negative: 4
Left Positive: 4
Left Total: 8
Right Negative: 0
Right Positive: 4
Right Total: 4
---
Position 8
Left Negative: 4
Left Positive: 5
Left Total: 9
Right Negative: 0
Right Pos

In [19]:
neg_left = (y == 0).cumsum()
pos_left = (y == 1).cumsum()
print('Negative Left', neg_left)
print('Positive Left', pos_left)

Negative Left [1 2 3 4 4 4 4 4 4 4 4 4]
Positive Left [0 0 0 0 1 2 3 4 5 6 7 8]


In [20]:
neg_total = neg_left[-1]
pos_total = pos_left[-1]
print('Negative Total', neg_total)
print('Positive Total', pos_total)

Negative Total 4
Positive Total 8


In [21]:
neg_right = neg_total - neg_left
pos_right = pos_total - pos_left
print('Negative Right', neg_right)
print('Positive Right', pos_right)

Negative Right [3 2 1 0 0 0 0 0 0 0 0 0]
Positive Right [8 8 8 8 7 6 5 4 3 2 1 0]


In [ ]:
def gini_search_CART(df):
    # initialize best_gini values
    best_gini = float('inf')
    best_split_value = None

    # Step 1: Sort the dataframe by the 'studied' column
    sorted_df = df.sort_values(by='studied')
    x = sorted_df['studied'].values
    y = sorted_df['passed'].values
    n = len(y)

    # Step 2: Initialize LEFT Node counts
    negative_left = (y == 0).cumsum()
    positive_left = (y == 1).cumsum()
    # print(negative_left)
    # print(positive_left)
    negative_total = negative_left[-1]
    positive_total = positive_left[-1]
    negative_right = negative_total - negative_left
    positive_right = positive_total - positive_left
    # print(negative_right)
    # print(positive_right)

    # Step 3: Move one sample from RIGHT to LEFT
    # Step 4: Update counts
    for i in range(n-1):
        
        left_total = negative_left[i] + positive_left[i]
        #print("Left total:",left_total)
        right_total = negative_right[i] + positive_right[i]
        #print("Right total:",right_total)
            
        left_negative = negative_left[i]
        left_positive = positive_left[i]
        right_negative = negative_right[i]
        right_positive = positive_right[i]    

        # Compute weighted gini
        left_gini = gini(left_positive, left_negative)
        right_gini = gini(right_positive, right_negative)
        weighted_gini = ((left_total/n) * left_gini) + ((right_total/n) * right_gini)
        #print("index:",i)
        print(f"Weighted Gini: {weighted_gini:.2f}")

        if weighted_gini < best_gini:
            best_gini = weighted_gini
            if x[i] == x[i+1]:
                continue
            else:
                best_split_value = (x[i] + x[i+1]) / 2
            print("Midpoint Threshold", best_split_value)
        
        print('---')
    
    return best_split_value, best_gini

In [23]:
gini_search_CART(df)

Weighted Gini: 0.36
Midpoint Threshold 1.5
---
Weighted Gini: 0.27
Midpoint Threshold 2.5
---
Weighted Gini: 0.15
Midpoint Threshold 3.5
---
Weighted Gini: 0.00
Midpoint Threshold 4.5
---
Weighted Gini: 0.13
---
Weighted Gini: 0.22
---
Weighted Gini: 0.29
---
Weighted Gini: 0.33
---
Weighted Gini: 0.37
---
Weighted Gini: 0.40
---
Weighted Gini: 0.42
---


(np.float64(4.5), np.float64(0.0))

## Gini Optimization Algorithm (Vectorization)

In [38]:
neg_left = (y == 0).cumsum()
pos_left = (y == 1).cumsum()
print('Negative Left', neg_left)
print('Positive Left', pos_left)

Negative Left [1 2 3 4 4 4 4 4 4 4 4 4]
Positive Left [0 0 0 0 1 2 3 4 5 6 7 8]


In [39]:
neg_total = neg_left[-1]
pos_total = pos_left[-1]
print('Negative Total', neg_total)
print('Positive Total', pos_total)

Negative Total 4
Positive Total 8


In [40]:
neg_right = neg_total - neg_left
pos_right = pos_total - pos_left
print('Negative Right', neg_right)
print('Positive Right', pos_right)

Negative Right [3 2 1 0 0 0 0 0 0 0 0 0]
Positive Right [8 8 8 8 7 6 5 4 3 2 1 0]


### Create Decision Tree Function

In [24]:
def gini(negative, positive):
    '''
    Compute the Gini impurity for a set of labels.
    negative: refers to number of negative labels
    positive: refers to number of positive labels
    '''
    total = negative + positive
    p0 = negative / total
    p1 = positive / total

    gini = 1 - (p0**2 + p1**2)
    return gini

In [ ]:
def best_gini_split(df):
    # initialize best_gini values
    best_gini = float('inf')
    best_split_value = None

    # Step 1: Sort the dataframe by the 'studied' column
    sorted_df = df.sort_values(by='studied')
    x = sorted_df['studied'].values
    y = sorted_df['passed'].values
    n = len(y)

    # Step 2: Initialize LEFT Node counts
    negative_left = (y == 0).cumsum()
    positive_left = (y == 1).cumsum()
    # print(negative_left)
    # print(positive_left)
    negative_total = negative_left[-1]
    positive_total = positive_left[-1]
    negative_right = negative_total - negative_left
    positive_right = positive_total - positive_left
    # print(negative_right)
    # print(positive_right)

    # Step 3: Move one sample from RIGHT to LEFT
    # Step 4: Update counts
    for i in range(n-1):
        
        left_total = negative_left[i] + positive_left[i]
        #print("Left total:",left_total)
        right_total = negative_right[i] + positive_right[i]
        #print("Right total:",right_total)
            
        left_negative = negative_left[i]
        left_positive = positive_left[i]
        right_negative = negative_right[i]
        right_positive = positive_right[i]    

        # Compute weighted gini
        left_gini = gini(left_positive, left_negative)
        right_gini = gini(right_positive, right_negative)
        weighted_gini = ((left_total/n) * left_gini) + ((right_total/n) * right_gini)
        #print("index:",i)
        #print(f"Weighted Gini: {weighted_gini:.2f}")

        if weighted_gini < best_gini:
            best_gini = weighted_gini
            if x[i] == x[i+1]:
                continue
            else:
                best_split_value = (x[i] + x[i+1]) / 2
            #print("Midpoint Threshold", best_split_value)
        
        #print('---')
    
    return best_split_value, best_gini

In [26]:
def built_tree(df):

    if df.empty:
        return None

    # If there's only one row left, return its label
    if len(df) <= 1:
        print(f"Single row subset found with label: {int(df['passed'].iloc[0])}")
        return int(df['passed'].iloc[0])
    
    # If all labels are identical, return that label
    # nunique() returns the number of unique values in the 'passed' column
    if df['passed'].nunique() == 1: 
        print(f"Pure subset found with label: {int(df['passed'].iloc[0])}")
        return int(df['passed'].iloc[0])


    # Find the best split based on Gini impurity
    threshold_studied, best_gini = best_gini_split(df)
    print(f"Best split value: {threshold_studied}, Gini after split: {best_gini:.2f}")

    # Split the dataset into two subsets based on the threshold value of 'studied'
    left_subset = df[df['studied'] <= threshold_studied]
    right_subset = df[df['studied'] > threshold_studied]

    print(f"Left subset:\n{left_subset}\n")
    print(f"Right subset:\n{right_subset}\n")

    # Create a dictionary to represent the current node
    clf = {'threshold_studied': float(threshold_studied)}

    # Recursively build the left and right subtrees
    clf['left'] = built_tree(left_subset)
    clf['right'] = built_tree(right_subset)

    return clf

### Build Tree 

We build a tree using current data frame.

In [27]:
df

,studied,passed
0,1,0
1,2,0
2,3,0
3,4,0
4,5,1
5,6,1
6,7,1
7,8,1
8,9,1
9,10,1


In [28]:
clf = built_tree(df)

Best split value: 4.5, Gini after split: 0.00
Left subset:
   studied  passed
0        1       0
1        2       0
2        3       0
3        4       0

Right subset:
    studied  passed
4         5       1
5         6       1
6         7       1
7         8       1
8         9       1
9        10       1
10       11       1
11       12       1

Pure subset found with label: 0
Pure subset found with label: 1


In [29]:
clf


{'threshold_studied': 4.5, 'left': 0, 'right': 1}

In [30]:
# print tree structure in json format
print(json.dumps(clf, indent=8))


{
        "threshold_studied": 4.5,
        "left": 0,
        "right": 1
}


In [31]:
# print tree structure with guided lines
def print_tree(node, indent="    "):
    if isinstance(node, dict):
        print(f"{indent}Threshold studied: {node['threshold_studied']}")
        print(f"{indent}├─ Left Node: <={node['threshold_studied']}")
        print_tree(node['left'], indent + "│   ")
        print(f"{indent}└─ Right Node: >{node['threshold_studied']}")
        print_tree(node['right'], indent + "    ")
    else:
        print(f"{indent}Passed: {node}")


In [32]:
print_tree(clf)

    Threshold studied: 4.5
    ├─ Left Node: <=4.5
    │   Passed: 0
    └─ Right Node: >4.5
        Passed: 1


### Prediction / Inference

To make a prediction we pass the prediction to the tree until we reach a leaf.

In [33]:
# make prediction function
def predict(clf, x):
    node = clf  # start at root of the tree

    # while we are still at a decision node (dictionary)
    while isinstance(node, dict):

        # get the split threshold stored at this node
        threshold = node['threshold_studied']

        # go left if x is smaller or equal to threshold
        if x <= threshold:
            print(f"Going left: {x} <= {threshold}")
            node = node['left']
        # otherwise go right
        else:
            print(f"Going right: {x} > {threshold}")
            node = node['right']

    # when we reach a leaf (not a dict), return the prediction
    return node

In [34]:
number_of_hours_studied = 5
prediction = predict(clf, number_of_hours_studied)
print(f"Prediction for student who studied {number_of_hours_studied} hours: {prediction}")

Going right: 5 > 4.5
Prediction for student who studied 5 hours: 1


## Scikit-Learn Compare

In the following, we will use Scikit-Learn to compare the tree. 

In [35]:
from sklearn.tree import DecisionTreeClassifier

sklearn_clf = DecisionTreeClassifier(criterion='gini')

sklearn_clf.fit(df[['studied']], df['passed'])

,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.",'gini'
,"splitter splitter: {""best"", ""random""}, default=""best""The strategy used to choose the split at each node. Supportedstrategies are ""best"" to choose the best split and ""random"" to choosethe best random split.",'best'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: int, float or {""sqrt"", ""log2""}, default=NoneThe number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... note:: The search for a split does not stop until at least one valid partition of the node samples is found, even if it requires to effectively inspect more than ``max_features`` features.",None
,"random_state random_state: int, RandomState instance or None, default=NoneControls the randomness of the estimator. The features are alwaysrandomly permuted at each split, even if ``splitter`` is set to``""best""``. When ``max_features < n_features``, the algorithm willselect ``max_features`` at random at each split before finding the bestsplit among them. But the best found split may vary across differentruns, even if ``max_features=n_features``. That is the case, if theimprovement of the criterion is identical for several splits and onesplit has to be selected at random. To obtain a deterministic behaviourduring fitting, ``random_state`` has to be fixed to an integer.See :term:`Glossary <random_state>` for details.",None
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow a tree with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsample

In [36]:
# get tree structure in json format
from sklearn.tree import export_text
tree_rules = export_text(sklearn_clf, feature_names=['studied'])
print(tree_rules)

|--- studied <= 4.50
|   |--- class: 0
|--- studied >  4.50
|   |--- class: 1



In [37]:
# make prediction function using sklearn
df_numbers_of_hours_studied = pd.DataFrame({'studied': [number_of_hours_studied]}) 
# need to convert to df as original data during training is in df format

sklearn_prediction = sklearn_clf.predict(df_numbers_of_hours_studied)
print(f"Prediction for student who studied {number_of_hours_studied} hours: {sklearn_prediction}")

Prediction for student who studied 5 hours: [1]


## End